# `pyfbs.nonlinearFBS` — Module Structure & Implementation

**A multi-harmonic-balance (HBM) nonlinear FBS solver ported into pyFBS from
[pyhbm](https://github.com/tiagomrns/pyhbm).**

Port baseline: pyhbm branch `vandcard_DLFT`, commit `462a081` (see the header of
`core.py` / `nonlinear_method.py`). Brought in by pyFBS commit `b443c49`
*"copied and trimmed pyhbm code"*.

This notebook documents the **whole implemented code structure** of the module:
every file, the classes they contain, the math each one realises, how the pieces
compose into a solve, and a final section on **what changed since the fork**.



---
## 1 · What the module computes

Given two (or more) substructures described **only by their FRFs / admittances**
$\mathbf{Y}(\omega)$ and coupled through a (generally nonlinear) interface force,
the module traces the **periodic steady-state response** of the assembly as a
function of excitation frequency — the *nonlinear forced-response curve* (NFRC).

The method is **Harmonic Balance + Alternating Frequency–Time (AFT)** for the
nonlinearity, wrapped in **pseudo-arclength numerical continuation** so the branch
can be followed through turning points (the "bent" resonances of hardening /
contact nonlinearities).

Nonlinearities supported out of the box:

| Strategy | Physics |
|---|---|
| `AFT` | any smooth interface force $f(u_{rel}, \dot u_{rel})$ — cubic/Duffing springs, nonlinear damping, … |
| `DLFTContact` | unilateral (one-sided) normal contact via Dynamic Lagrangian Frequency–Time |
| `DLFTFriction` | unilateral contact **+** Coulomb friction (Nacivet et al., JSV 2003) |

FRF sources supported:

| Provider | FRF source |
|---|---|
| `NumericalFRF` | mode-superposition synthesis from a **real modal basis** (e.g. ANSYS modes) |
| `ModalVPFRF` | virtual-point admittance with the **VPT folded into the mode shapes** |
| `ExperimentalFRF` | cubic-spline interpolation of **measured** frequency-domain data |


---
## 2 · Package layout — file map

```
pyfbs/nonlinearFBS/
├── __init__.py                      public API (re-exports every class below)
├── core.py                          SolutionSet, HarmonicBalanceMethod   ← the driver
├── dynamical_system.py             FBS_System                            ← user base class
├── frequency_domain.py             Fourier machinery (FFT ↔ coefficients)
├── frf_provider.py                 FRFProvider, NumericalFRF, ModalVPFRF, ExperimentalFRF
├── nonlinear_method.py             NonlinearMethod, AFT, DLFTContact, DLFTFriction
├── hbm_problems.py                 FBSProblem                            ← assembles R, J, dR/dω
├── numerical_continuation/
│   ├── corrector_step.py           NewtonRaphson + corrector parameterizations
│   └── predictor_step.py           tangent predictors + step-length adaptation
└── examples/                        runnable validation cases (see §10)
    ├── duffing_FBS_2DoF/            nonlinear 2-DOF Duffing (analytic FRF)
    ├── rod_example/                 single FE rod, smooth nonlinearity
    ├── two_rod_example/            two FE rods, vibro-impact (DLFT)
    ├── sdof_vibroimpact_validation/ SDOF against a wall (DLFTContact)
    ├── testbench_linearSpring/     pyFBS lab testbench, linear bushing (LM-FBS verification)
    └── testbench_cubicSpring/      pyFBS lab testbench, cubic bushing (true NFRC)
```

| File | Role in one line |
|---|---|
| `dynamical_system.py` | **what you subclass**: declares the coupling matrix $\mathbf{B}$ and the interface force |
| `frf_provider.py` | **where the FRF comes from**: turns $\omega$ into per-harmonic admittance blocks $\mathbf{Y}_n$ |
| `nonlinear_method.py` | **how the nonlinearity is handled**: AFT / DLFT strategy objects |
| `hbm_problems.py` | **the residual**: glues system + FRF + nonlinearity into $R,\,J,\,\partial R/\partial\omega$ |
| `frequency_domain.py` | **the maths plumbing**: Fourier ↔ time, real/imag (RI) packing, FFT Jacobians |
| `numerical_continuation/` | **the path-follower**: Newton corrector + tangent predictor + step control |
| `core.py` | **the driver**: `HarmonicBalanceMethod.solve_and_continue` runs the whole loop |


---
## 3 · Architecture — how the pieces compose

The design is **compositional** (strategy objects), not a deep inheritance tree.
A problem is built by plugging three independent objects into `FBSProblem`, which
is then handed to the `HarmonicBalanceMethod` driver:

```
        ┌─────────────────────────── HarmonicBalanceMethod ───────────────────────────┐
        │  solve_and_continue():  predictor → corrector (Newton) → step adaptation     │
        │                                                                              │
        │     predictor_step.py            corrector_step.py                           │
        │     (tangent direction)          (NewtonRaphson + parameterization)          │
        └───────────────────────────────────┬──────────────────────────────────────────┘
                                             │ calls R, J, dR/dω on
                                             ▼
                                   ┌───────  FBSProblem  ───────┐         (hbm_problems.py)
                                   │  R = Q_rel + Y_r F_nl − F_adm │
                                   └───┬──────────┬──────────┬────┘
                       ode (system)    │          │          │  frf_provider
                                       ▼          ▼          ▼
                               FBS_System   NonlinearMethod   FRFProvider
                            (dynamical_     (AFT / DLFT*)     (Numerical /
                             system.py)                       ModalVP /
                                                              Experimental FRF)
```

Read it as three plug-in slots:

```python
problem = FBSProblem(
    fbs          = my_system,        # FBS_System  subclass  (B, interface force)
    frf_provider = my_frf,           # FRFProvider subclass  (Y(ω) source)
    method       = AFT(),            # NonlinearMethod        (AFT / DLFTContact / DLFTFriction)
)
solver = HarmonicBalanceMethod(harmonics=[1, 3, 5], freq_domain_ode=problem, ...)
solution_set = solver.solve_and_continue(...)
```

Each slot is swappable: the same `FBS_System` can be driven with measured FRFs
(`ExperimentalFRF`) or synthesised ones (`ModalVPFRF`) without touching the
physics; the same FRF can carry a smooth spring (`AFT`) or contact (`DLFTContact`).


---
## 4 · The FBS-HBM residual (the maths the module solves)

Substructures are described by the **uncoupled, block-diagonal** admittance
$\mathbf{Y}^{A|B}(\omega) = \mathrm{diag}(\mathbf{Y}^A, \mathbf{Y}^B, \dots)$ and
coupled by an interface force $\mathbf{F}^{nl}$ acting on the **relative interface
displacement** $u_{rel} = \mathbf{B}\,u$, where $\mathbf{B}\in\mathbb{R}^{n_{int}\times d_{total}}$
is the signed-Boolean coupling matrix.

The HBM unknown is $\mathbf{Q}_{rel}$, the Fourier coefficients of $u_{rel}$. Per
harmonic the substructure EOM $\mathbf{Q}_n = \mathbf{Y}_n(\mathbf{F}^{ext}_n - \mathbf{B}^T\boldsymbol{\Lambda}_n)$
projected by $\mathbf{B}$ gives the **residual** (with $\mathbf{Y}_r = \mathbf{B}\,\mathbf{Y}\,\mathbf{B}^T$
and $\mathbf{F}_{adm} = \mathbf{B}\,\mathbf{Y}\,\mathbf{F}^{ext}$):

$$\boxed{\;\mathbf{R} = \mathbf{Q}_{rel} + \mathbf{Y}_r\,\mathbf{F}^{nl}(\mathbf{Q}_{rel}) - \mathbf{F}_{adm} = \mathbf{0}\;}$$

**Jacobian** (state) and **frequency derivative** (for the continuation tangent):

$$\frac{\partial\mathbf{R}}{\partial\mathbf{Q}_{rel}} = \mathbf{I} + (\mathbf{Y}_r)_{RI}\,\mathbf{J}^{nl}_{RI},
\qquad
\frac{\partial\mathbf{R}}{\partial\omega}\bigg|_{\mathbf{Q}_{rel}}
= \mathbf{B}\,\frac{d\mathbf{Y}}{d\omega}\big(\mathbf{B}^T\mathbf{F}^{nl} - \mathbf{F}^{ext}\big)
+ \mathbf{Y}_r\,\frac{\partial\mathbf{F}^{nl}}{\partial\omega}.$$

Everything is solved in **real–imaginary (RI) packed form**: a complex per-harmonic
quantity $c$ becomes $[\,\mathrm{Re}\,c;\ \mathrm{Im}\,c\,]$, and a complex operator
becomes the real block $\begin{bmatrix}\mathrm{Re} & -\mathrm{Im}\\ \mathrm{Im} & \mathrm{Re}\end{bmatrix}$
(`block_diag_stack_to_RI` in `frequency_domain.py`).

This is exactly what `FBSProblem.compute_residue_RI / _jacobian_of_residue_RI /
_derivative_wrt_omega_RI` implement (§8).


---
## 5 · `dynamical_system.py` — `FBS_System`

The base class **you subclass** to define a problem. It is deliberately thin: it
carries the coupling matrix and the AFT sample count, and declares the *semantic
interface* the solver calls.

| Attribute | Shape | Meaning |
|---|---|---|
| `B_coupling` | $(n_{int}, d_{total})$ | signed-Boolean coupling; each row picks an interface DOF pair ($+1/-1$) |
| `sample_number` | int | number of AFT time samples (must resolve the highest harmonic) |
| `omega_ref` | float | reference frequency used to scale the continuation metric (default 1.0) |

> Note: $n_{int}$ (= rows of $\mathbf{B}$) and $d_{total}$ (= FRF DOF count) are **not**
> stored here — `FBSProblem` derives them from `B_coupling` and the FRF provider.

**Subclasses must implement** (the "semantic" interface, operating on the relative
interface gap $u_{rel}$):

```python
def external_term(self, tau)                         -> (Nt, d_total, 1)   # F_ext(τ)
def interface_force(self, u_rel, u_rel_dot, tau)     -> (Nt, n_int, 1)     # F_nl
def jacobian_interface_force(self, u_rel, u_rel_dot, tau)      -> (Nt, n_int, n_int)
def jacobian_interface_force_qdot(self, u_rel, u_rel_dot, tau) -> (Nt, n_int, n_int)
```

**Framework wrappers** (do *not* override) adapt these to the generic names the AFT
machinery expects — they just forward to the semantic methods:

```python
def nonlinear_term(self, u_rel, u_rel_dot, tau):
    return self.interface_force(u_rel, u_rel_dot, tau)
def jacobian_nonlinear_term(self, u_rel, u_rel_dot, tau):
    return self.jacobian_interface_force(u_rel, u_rel_dot, tau)
def jacobian_nonlinear_term_qdot(self, u_rel, u_rel_dot, tau):
    return self.jacobian_interface_force_qdot(u_rel, u_rel_dot, tau)
```

A minimal cubic (Duffing) bushing on a 6-DOF virtual-point gap — straight from
`examples/testbench_cubicSpring`:

```python
class TestbenchCubicSpring(FBS_System):
    def interface_force(self, u_rel, udot_rel, tau):           # f = k (x + α x³)
        k, a = self.k_diag[None,:,None], self.alpha_diag[None,:,None]
        return k * (u_rel + a * u_rel**3)                       # (Nt, 6, 1)
    def jacobian_interface_force(self, u_rel, udot_rel, tau):  # df/dx = k (1 + 3α x²)
        ...                                                     # diagonal (Nt, 6, 6)
    def jacobian_interface_force_qdot(self, u_rel, udot_rel, tau):
        return np.zeros((len(tau), 6, 6))                      # no velocity dependence
```


---
## 6 · `frequency_domain.py` — the Fourier plumbing

The transform layer that turns the AFT loop between **frequency** (Fourier
coefficients) and **time** (samples), and packs complex quantities into the real
RI form the Newton solver uses. No physics here — pure signal machinery, ported
essentially unchanged from pyhbm.

| Class / function | Responsibility |
|---|---|
| `Fourier` | holds the per-harmonic coefficient stack `(Nh, d, 1)`; `+`, `−`, scalar `*`, RI packing (`__array__`), `get_adimensional_time_derivative` ($\partial_\tau \leftrightarrow i n$) |
| `Fourier_Real` | real-signal specialisation: `new_from_time_series` (rFFT) and `compute_time_series` (irFFT) — the **F→T / T→F** of the AFT loop |
| `FourierOmegaPoint` | a continuation point: a `Fourier` **plus** its frequency $\omega$, **plus a large bag of per-point caches** ($Y$, $dY$, $B Y$, $Y_r$, $F_{adm}$, contact mask, …) so each Newton step recomputes nothing |
| `JacobianFourier` / `JacobianFourier_Real` | the **Hankel/Toeplitz** AFT Jacobian: builds $\partial \widehat{f}/\partial\widehat{q}$ blocks from the time-domain tangent via FFT, returned as four real sub-blocks `RR, RI, IR, II` |
| `block_diag_stack_to_RI(blocks)` | embeds a `(Nh, a, b)` block-diagonal complex operator into the dense real $\begin{bmatrix}\mathrm{Re}&-\mathrm{Im}\\\mathrm{Im}&\mathrm{Re}\end{bmatrix}$ form — only ever applied at *interface* size |

`FourierOmegaPoint` is where the **caching strategy** lives. Within one Newton step
the residual, Jacobian and $\partial R/\partial\omega$ all need the same $\mathbf{Y}$,
$\mathbf{B}\mathbf{Y}$, $\mathbf{Y}_r$, $\mathbf{F}_{adm}$, … so each is computed
once and stored on the point:

```python
self.Y_cache = self.dY_cache = None          # Y(ω), dY/dω
self.BY_cache = self.Yr_cache = None         # B Y, Y_r = B Y Bᵀ  (per-harmonic stacks)
self.Fext_admr_cache = self.Zr_rhs = None    # F_adm = B Y F_ext, and the DLFT balancing solve
self.lambda_corrected = self.contact_mask = None   # DLFT corrected force + stick/slip/sep mask
```

> **DC-harmonic fix (`JacobianFourier_Real`).** With pyhbm's rFFT convention the DC
> bin $c_0 = N_t a_0$ has no factor of 2 (unlike $c_k$, $k\ge1$). The Hankel doubling
> $G_{n-m}+G_{n+m}$ over-counts the $m=0$ column by 2, so it is halved when harmonic 0
> is present. Found via a finite-difference Jacobian check in the vibro-impact example.


---
## 7 · `frf_provider.py` — where the admittance comes from

The **pyFBS-integration heart of the port** (see §11). An `FRFProvider` answers one
question: *given $\omega$ and the requested harmonics, return the admittance blocks
$\mathbf{Y}_n$ — restricted to the small set of output/input DOFs the solver actually
touches.*

```python
class FRFProvider(ABC):
    @abstractmethod
    def compute_FRF(self, omega, harmonics, out_dofs, in_dofs) -> array:   # (Nh, |out|, |in|)
    @abstractmethod
    def compute_FRF_derivative(self, omega, harmonics, out_dofs, in_dofs, Y_cache) -> array:
    @property
    @abstractmethod
    def n_dofs(self) -> int:        # full DOF count that out_dofs / in_dofs index into
```

The `out_dofs` / `in_dofs` arguments are the **DOF-reduction** mechanism: `FBSProblem`
passes only the interface ∪ excitation DOFs, so a provider synthesises just that small
block per evaluation — cost is **flat in mesh size**.

| Provider | FRF source | $dY/d\omega$ |
|---|---|---|
| `NumericalFRF` | mode-superposition receptance from a **real modal basis**; `from_modal(Ω, Φ, ζ)` or `from_ansys_model(model, ζ)`. Never eigensolves, never assembles $M,C,K$. Uses pyFBS `Model.custom_frf_synth`. | analytic modal-sum derivative (`custom_dfrf_domega_synth`) |
| `ModalVPFRF` | **virtual-point** admittance with the VPT folded into the mode shapes: $\mathbf{Y}_{vp} = (\mathbf{T}_u\Phi_c)\,D(\omega)\,(\mathbf{T}_f^T\Phi_i)^T$. `from_substructures([(model, vpt, df_chn, df_imp), …], ζ)` block-diagonal-stacks the substructures. Synthesises **directly in VP space** — physical $\mathbf{Y}$ never assembled. | analytic modal-sum derivative |
| `ExperimentalFRF` | cubic-spline interpolation of measured `Y(ω_frf)`; conjugate symmetry $\mathbf{Y}(-\omega)=\overline{\mathbf{Y}(\omega)}$. **Project-then-interpolate**: only the requested `Y[:, out, in]` channels are splined, cached per DOF set. | central finite difference |

The `ModalVPFRF` algebra (why the VPT commutes through the modal sum): the VPT is a
constant spatial projection and the modal denominator $D(\omega)$ is diagonal in the
modes, so

$$\mathbf{Y}_{vp} = \mathbf{T}_u\big(\Phi_c\,D(\omega)\,\Phi_i^T\big)\mathbf{T}_f
= \underbrace{(\mathbf{T}_u\Phi_c)}_{\Psi_c}\,D(\omega)\,\underbrace{(\mathbf{T}_f^T\Phi_i)^T}_{\Psi_i^T}.$$

Only the VP-projected participation matrices $\Psi_c,\Psi_i$ are stored; the poles
$\Omega$ and modal damping are the unchanged physical ones (the VPT moves mode shapes,
not poles). Cost is $O(n_{modes}\cdot N_{vp}^2)$ — flat in mesh size, and `n_dofs == N_vp`
matches the FBS Boolean column count so a plain `FBSProblem` consumes it unchanged.


---
## 8 · `nonlinear_method.py` — the nonlinearity strategies

A `NonlinearMethod` supplies the three quantities `FBSProblem` needs from the
nonlinearity, all in RI form. `bind(problem)` lets a method capture problem-level
context (interface size, $\mathbf{B}$, FRF caches).

```python
class NonlinearMethod(ABC):
    def bind(self, problem): ...                     # capture context (default no-op)
    def compute_F_int(self, x, ode)            -> (Nh·d, 1) complex     # F_nl
    def compute_J_int_RI(self, x, ode)         -> (2Nh·d, 2Nh·d) real   # dF_nl/dx_r (RI)
    def compute_dF_int_domega_RI(self, x, ode) -> (2Nh·d, 1) real       # dF_nl/dω (RI)
```

### 8.1 `AFT` — Alternating Frequency–Time
For a smooth interface force. Each evaluation: inverse-FFT $\mathbf{Q}_{rel}\to u_{rel}(\tau)$,
evaluate `interface_force` in time, forward-FFT back to coefficients. The Jacobian is
assembled from the time-domain tangents (`jacobian_nonlinear_term` and the velocity part
`…_qdot`) through `JacobianFourier_Real`, with the $\dot u_{rel}=\omega\,u'_{rel}$ chain
rule folded in via a `kron(diag(harmonics), I)` column scaling.

### 8.2 `DLFTContact` — Dynamic Lagrangian, normal contact
Unilateral contact without solving an LCP. The contact force is **slaved** to
$\mathbf{Q}_{rel}$ by a prediction–correction at every residue evaluation:

$$\lambda_p = \mathrm{IDFT}\!\big[\mathbf{Z}_r(\mathbf{F}_{adm}-x_r)\big] + \varepsilon(x_r - g_0),
\qquad \lambda = \max(0,\lambda_p), \qquad \tilde\lambda = \mathrm{DFT}[\lambda].$$

Here $\mathbf{Z}_r = \mathbf{Y}_r^{-1}$ is the dynamic interface stiffness (solved
per-harmonic, batched). The Jacobian uses the time-domain contact mask
$m = (\lambda_p>0)$ through $\mathbf{J}_{mask}(\varepsilon\mathbf{I}-\mathbf{Z}_r)$.
Bound to an `FBSProblem`, it reads $\mathbf{B}$, $\mathbf{F}_{ext}$ and the admittance
caches through that reference.

### 8.3 `DLFTFriction` — normal contact **+** Coulomb friction
Same DLFT skeleton, but the corrector is a **sequential time-domain stick/slip/separation
sweep** (Nacivet, Pierre, Thouverez & Jézéquel, *JSV* 265(1), 2003). Interface DOFs are
grouped into per-node `[N, T…]` blocks (`n_dir = 1 + n_tangential`); each sample is
classified separation / stick / slip, with the analytical local tangent `Jloc` cached for
the Jacobian. `n_sweep > 1` iterates the sweep toward an exactly-periodic tangential force
(a refinement beyond the paper).


---
## 9 · `hbm_problems.py` — `FBSProblem` (the residual assembler)

The class that turns the three plug-ins into the functions the continuation needs.
Constructor wiring:

```python
self.d_int   = fbs.B_coupling.shape[0]   # n_int  (Newton unknowns / harmonic)
self.d_total = frf_provider.n_dofs       # full DOF count (from the FRF data, not M,C,K)
# DOF reduction: only interface ∪ excitation DOFs are ever synthesised
idof = nonzero columns of B ; xdof = nonzero rows of F_ext
self.in_dofs = union1d(idof, xdof)
self.B     = B_full[:, in_dofs]          # reduced coupling
self.F_ext = F_ext_full[:, in_dofs, :]   # reduced excitation
self.method.bind(self)
```

The three core methods implement §4 verbatim, all on **per-harmonic stacks** with
only interface-sized dense matrices ever formed:

```python
def compute_residue_RI(self, x):                       # R = Q_rel + Y_r F_nl − F_adm
    Q_rel = x.fourier.coefficients
    Fnl   = self.method.compute_F_int(x, self.ode).reshape(Q_rel.shape)
    R     = Q_rel + self._get_Yr(x) @ Fnl - self._get_Fadm(x)
    return concat([R.real.ravel; R.imag.ravel])

def compute_jacobian_of_residue_RI(self, x):           # J = I + (Y_r)_RI J_nl_RI
    return eye(N) + self._get_BYBT_RI(x) @ self.method.compute_J_int_RI(x, self.ode)

def compute_derivative_wrt_omega_RI(self, x):          # dR/dω
    dR = (self.B @ self._get_dY(x)) @ (self.B.T @ Fnl - self.F_ext) \
       + self._get_Yr(x) @ dF_nl_dω
    ...
```

Cached intermediates (all keyed on the `FourierOmegaPoint`, computed at most once):
$\mathbf{Y}$ → $\mathbf{B}\mathbf{Y}$ → $\mathbf{Y}_r=\mathbf{B}\mathbf{Y}\mathbf{B}^T$ →
$(\mathbf{Y}_r)_{RI}$, plus $\mathbf{F}_{adm}=\mathbf{B}\mathbf{Y}\mathbf{F}_{ext}$ and
$d\mathbf{Y}/d\omega$.

Post-processing maps the reduced solution back to **every physical DOF**:

```python
def compute_full_response(self, fourier, omega):       # one extra Y[all_dofs, in_dofs] call
    Q_full = Y_full @ (self.F_ext - self.B.T @ Fnl)     # (Nh, d_total, 1)
    return Fourier(Q_full)
```


---
## 10 · `numerical_continuation/` + `core.py` — the path-follower

### 10.1 `corrector_step.py`
| Class | Role |
|---|---|
| `NewtonRaphson` | the corrector. Damped Newton with **Armijo backtracking line search**, lazy **Jacobian reuse** (`jacobian_update_frequency`, `jacobian_reuse_delta_threshold`), and optional **column scaling** `D` for the extended $(x_r,\omega)$ solve so the $\omega$ column is $O(1)$ |
| `CorrectorParameterization` | base for the one scalar equation that pins Newton to a point on the curve |
| `OrthogonalParameterization` | correction lies in the hyperplane ⟂ to the tangent through the predicted point — robust through turning points |
| `ArcLengthParameterization` | Keller pseudo-arclength: corrected point on the sphere of radius = step size around the last solution |

### 10.2 `predictor_step.py`
| Class | Role |
|---|---|
| `TangentPredictorOne` / `…Two` / `…Robust` | null-space tangent of the Jacobian (kernel dim 1 / 2 / verified) |
| `TangentPredictorBordered` | Keller bordered system — stable when the Jacobian goes rank-deficient at limit points |
| `ExponentialAdaptation` / `BiExponentialAdaptation` | step length grows/shrinks toward a goal iteration count |
| `_metric_normalize` | normalises the tangent in the **scaled continuation metric** $\lVert D^{-1}v\rVert = 1$ |

### 10.3 `core.py` — the driver
`SolutionSet` accumulates `(fourier, omega, iterations, step_length)` along the branch.
`HarmonicBalanceMethod`:

- `update_dependencies(harmonics, sample_number)` — pushes the harmonic set into the
  `Fourier` / `JacobianFourier` class variables (must be called before solving);
- `solve_fixed_frequency(...)` — one Newton solve at fixed $\omega$ (the branch seed);
- `solve_and_continue(...)` — the **predictor → corrector → step-adaptation** loop:
  predict a tangent step, correct with the parameterised Newton solve, accept/retry with
  adapted step length, append, and stop on frequency-range exit / predictor failure /
  solver stall / max solutions.

The continuation runs **directly in physical rad/s**; `Dscale` (built from
`omega_ref`) only conditions the linear algebra — the curve itself is un-rescaled.


---
## 11 · End-to-end usage (copy-runnable)

Condensed from `examples/testbench_linearSpring/testbench_linearSpringCoupling.py`:
the **same coupling** is run through the solver and overlaid on the closed-form
LM-FBS receptance to verify the pipeline. (Needs the ANSYS lab-testbench data; left
unexecuted here.)


In [1]:
import numpy as np
import os, sys
if not os.getcwd().endswith("testbench_linearSpring"):
    os.chdir("examples/testbench_linearSpring")   # Beispiel-Modul + ./lab_testbench Daten
sys.path.insert(0, os.getcwd())
from pyfbs.nonlinearFBS import (
    Fourier_Real, FourierOmegaPoint, FBSProblem, ExperimentalFRF, ModalVPFRF, AFT,
    HarmonicBalanceMethod,
)
from pyfbs.nonlinearFBS.numerical_continuation.corrector_step import ArcLengthParameterization
from pyfbs.nonlinearFBS.numerical_continuation.predictor_step import TangentPredictorBordered
from dynamical_system import build_testbench_data, TestbenchLinearSpring, N_IF

# 1) data: ANSYS testbench -> block-diagonal admittance Y = diag(Y_A, Y_B)
data   = build_testbench_data(k_trans=1e6, k_rot=1e3, c_trans=1e2, c_rot=1e2, f_resolution=0.1)
system = TestbenchLinearSpring(data, F0=1.0)              # FBS_System subclass

# 2) tell the Fourier layer which harmonics to balance (BEFORE solving)
HARMONICS = [1]
HarmonicBalanceMethod.update_dependencies(HARMONICS, system.sample_number)

# 3) pick an FRF source: measured-grid spline ...
provider = ExperimentalFRF(data["omega"], data["Y"])
# ... or VPT folded into the FE modes, synthesised at the exact n·ω:
# provider = ModalVPFRF.from_substructures(
#     [(data["MK_A"], data["vpt_A"], data["df_chn_A"], data["df_imp_A"]),
#      (data["MK_B"], data["vpt_B"], data["df_chn_B"], data["df_imp_B"])],
#     modal_damping=data["modal_damping"])

# 4) assemble the problem (system + FRF + nonlinearity) and the driver
problem = FBSProblem(system, provider, AFT())
solver  = HarmonicBalanceMethod(
    harmonics=HARMONICS, freq_domain_ode=problem,
    corrector_parameterization=ArcLengthParameterization,
    predictor=TangentPredictorBordered)

# 5) seed + arc-length continuation across the band
w_lo, w_hi = 2*np.pi*0.0, 2*np.pi*2000.0
ig = FourierOmegaPoint.zero_amplitude(dimension=N_IF, omega=w_hi)
rd = FourierOmegaPoint.new_from_first_harmonic(np.zeros((N_IF, 1), complex), omega=-1.0)
ss = solver.solve_and_continue(
    initial_guess=ig, initial_reference_direction=rd,
    maximum_number_of_solutions=10000, angular_frequency_range=[w_lo, w_hi],
    solver_kwargs={"maximum_iterations": 300, "absolute_tolerance": 1e-6},
    step_length_adaptation_kwargs={"base": 2.0, "initial_step_length": 1.0,
                                   "maximum_step_length": 10.0, "minimum_step_length": 1e-4,
                                   "goal_number_of_iterations": 3})

# 6) post-process: reconstruct full physical response per converged point
freqs_Hz = np.array(ss.omega) / (2*np.pi)
recept = []
for four, w in zip(ss.fourier, ss.omega):
    full = problem.compute_full_response(four, w)
    Fourier_Real.compute_time_series(full)
    recept.append(float(np.max(np.abs(full.time_series[:, system.out_full, 0]))))


  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 9808.94it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:00<00:00, 14090.61it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 37888.93it/s]

(6790, 3)
(6496, 3)


(21, 10)
(18, 10)
(21, 11)
(18, 10)


C:\Users\yanni\Desktop\TUM\Bachelorarbeit\Fast_numerical_solution_for_nonlinear_FE_dynamics\pyFBS_clone\pyFBS\pyfbs\nonlinearFBS\frf_provider.py:266: UserWarning: omega outside FRF data range [0, 12565.7423]. Extrapolating.
  warnings.warn(


progress 12.580 % 	iterations 5 	Δω -1.00e+00
Terminate: outside frequency range after 1259 solutions
Total solving time: 1.3639044761657715 seconds


---
## 12 · `examples/` — validation cases

Each folder is a self-contained `python main.py` script (system definition in
`dynamical_system.py`, driver in `main.py`).

| Example | System | Nonlinearity | FRF source | Validates |
|---|---|---|---|---|
| `duffing_FBS_2DoF` | 2 single-DOF oscillators | cubic spring / nonlinear damping | analytic | NFRC vs 1st-order HBM reference |
| `rod_example` | one clamped-free FE rod | smooth | modal | basic FE-driven AFT path |
| `two_rod_example` | two FE rods across a gap | vibro-impact (DLFT) | numerical / experimental | contact NFRC, optional external CSV overlay |
| `sdof_vibroimpact_validation` | SDOF mass against a wall | unilateral contact (`DLFTContact`) | — | DLFT contact + the DC-Jacobian fix |
| `testbench_linearSpring` | pyFBS lab testbench A+B | linear bushing | `ExperimentalFRF` **and** `ModalVPFRF` | solver vs closed-form **LM-FBS** receptance |
| `testbench_cubicSpring` | pyFBS lab testbench A+B | cubic (hardening) bushing | `ExperimentalFRF` / `ModalVPFRF` | true bent NFRC on real testbench data |

The two `testbench_*` cases are the pyFBS-specific milestones: they exercise the real
lab-testbench FE models, the VPT, and both FRF-provider pipelines end to end.


---
## 13 · What changed since the fork

Baseline: the module was **copied and trimmed** from pyhbm (`vandcard_DLFT @ 462a081`).
The Fourier machinery (`frequency_domain.py`), the continuation skeleton, and the
DLFT/AFT strategies are ported largely as-is; the changes concentrate where the solver
meets **pyFBS's own FRF / VPT / FE infrastructure**.

### 13.1 Scope — trimmed to the FBS path
Dropped from the port (present in upstream pyhbm): `FRFProblem` (single-subsystem
solver), `FirstOrderODE` / `SecondOrderODE` base classes, the whole legacy
`FrequencyDomain*` / `FrequencyBasedSubstructuring*` class hierarchy, and the
`stability/` (Floquet, bifurcation), `validation/` (`TimeDomainValidator`), and `io/`
(`plot_FRF`, `save_solution_set`) subpackages. `__init__.py` exports only the FBS API.

### 13.2 `frf_provider.py` — reworked around pyFBS (the biggest change)
| | upstream pyhbm | pyFBS port |
|---|---|---|
| interface | `compute_FRF(ω, harmonics, d)` returns full `(Nh, d, d)` | `compute_FRF(ω, harmonics, out_dofs, in_dofs)` + abstract `n_dofs` → **DOF reduction** (synthesise only the touched block) |
| `NumericalFRF` | takes $(M,C,K)$; inverts $Z_n=-(n\omega)^2M+in\omega C+K$ per harmonic | takes a **real modal basis** (`from_modal` / `from_ansys_model`); mode-superposition via pyFBS `Model.custom_frf_synth`; never assembles $M,C,K$, never eigensolves |
| `ModalVPFRF` | — | **new**: virtual-point admittance with the VPT folded into the mode shapes (`from_substructures`), synthesised directly in VP space |
| `ExperimentalFRF` | splines the whole `Y` up front | **project-then-interpolate**: per-DOF-set spline cache, flat in measured DOF count |

### 13.3 `dynamical_system.py` — `FBS_System` slimmed
Removed the `mass_matrix` / `damping_matrix` / `stiffness_matrix`, `dimension`,
`polynomial_degree` attributes (the system is now defined by its **FRF**, not $M,C,K$).
The class keeps only `B_coupling`, `sample_number`, `omega_ref` and the semantic
interface-force methods. (Some legacy examples still set the old attributes harmlessly.)

### 13.4 `hbm_problems.py` — `FBSProblem`
`d_total` now comes from `frf_provider.n_dofs` instead of `fbs.mass_matrix.shape[0]`;
added the **`in_dofs` DOF-reduction** (reduced `B` and `F_ext`), the `Dscale` /
`omega_ref` continuation-metric scaling, and `compute_full_response` requesting
`Y[all_dofs, in_dofs]` to recover every physical DOF from the reduced solve.

### 13.5 `core.py` + `numerical_continuation/`
`HarmonicBalanceMethod`'s constructor dropped the `first_order_ode` / `second_order_ode`
paths (FBS-only) and gained `Dscale` column scaling + `reference_force_level`. The
predictors and parameterizations gained the **scaled metric** (`_metric_normalize`,
`dinv2`/`dscale` arguments); `NewtonRaphson` gained `column_scale`, Armijo line search and
lazy Jacobian reuse; and the long-standing `TangentPredictorRobust` `step_length` bug from
upstream was fixed.

### 13.6 `nonlinear_method.py`
Ported; adapted to the per-harmonic stacks and to bind against `FBSProblem` (AFT now sizes
its Jacobian from `problem.d_int` rather than `ode.dimension`; DLFT methods read the
`Y_r` / `F_adm` / `dY` caches through the bound problem).

### 13.7 New examples
`testbench_linearSpring/` and `testbench_cubicSpring/` are new pyFBS examples built on the
real lab-testbench FE models + VPT, exercising both the `ExperimentalFRF` and `ModalVPFRF`
pipelines (the linear case cross-checks against the closed-form LM-FBS coupling).
